
# Keyword vs. Semantic Search 

## Imports <a name="im"></a>

In [3]:
import re
import pandas as pd
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

/Users/kvarada/EL/workshops/GIRLsmarts-workshop/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
docs = [
    "My puppy loves chasing tennis balls.",
    "My dog enjoys long walks in the park.",
    "Cats sleep for most of the day.",
    "The kitten climbed the tree.",
    "I baked chocolate chip cookies yesterday.",
    "This recipe makes delicious brownies.",
    "Vancouver received heavy rainfall today.",
    "It might rain tomorrow.",
    "Taylor Swift released a new album.",
    "I listened to some great music.",
    "I rode my bicycle to school.",
    "My bike has a flat tire.",
    "The physician recommended more exercise.",
    "The doctor prescribed antibiotics.",
    "Penguins cannot fly.",
]

def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())

tokenized_docs = [tokenize(doc) for doc in docs]

bm25 = BM25Okapi(tokenized_docs)

def bm25_search(query, top_k=5):
    tokenized_query = tokenize(query)
    scores = bm25.get_scores(tokenized_query)
    ranked_indices = scores.argsort()[::-1][:top_k]

    return [(idx + 1, docs[idx], scores[idx]) for idx in ranked_indices if scores[idx] > 0]


model = SentenceTransformer("all-MiniLM-L6-v2")
doc_embeddings = model.encode(docs)

def semantic_search(query, top_k=5):
    query_embedding = model.encode([query])
    scores = cosine_similarity(query_embedding, doc_embeddings)[0]
    ranked_indices = scores.argsort()[::-1][:top_k]

    return [(idx + 1, docs[idx], scores[idx]) for idx in ranked_indices]

def compare_search(query, top_k=5):
    print(f"\nQUERY: {query!r}\n")

    print("Keyword search results")
    bm25_results = bm25_search(query, top_k=top_k)

    if bm25_results:
        for doc_id, doc, score in bm25_results:
            print(f"{doc_id}. {doc}  (score = {score:.2f})")
    else:
        print("No keyword matches.")

    print("\nSemantic search results")
    semantic_results = semantic_search(query, top_k=top_k)

    for doc_id, doc, score in semantic_results:
        print(f"{doc_id}. {doc}  (score = {score:.2f})")

In [5]:
bm25 = BM25Okapi(tokenized_docs)

def bm25_search(query, top_k=5):
    tokenized_query = tokenize(query)
    scores = bm25.get_scores(tokenized_query)
    ranked_indices = scores.argsort()[::-1][:top_k]

    return [(idx + 1, docs[idx], scores[idx]) for idx in ranked_indices if scores[idx] > 0]

In [6]:
model = SentenceTransformer("all-MiniLM-L6-v2")
doc_embeddings = model.encode(docs)

def semantic_search(query, top_k=5):
    query_embedding = model.encode([query])
    scores = cosine_similarity(query_embedding, doc_embeddings)[0]
    ranked_indices = scores.argsort()[::-1][:top_k]

    return [(idx + 1, docs[idx], scores[idx]) for idx in ranked_indices]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7886.62it/s]


In [12]:
def compare_search(query, top_k=5):
    print(f"\nQUERY: {query!r}\n")

    print("Keyword search results")
    bm25_results = bm25_search(query, top_k=top_k)

    if bm25_results:
        for doc_id, doc, score in bm25_results:
            print(f"{doc_id}. {doc}  (score = {score:.2f})")
    else:
        print("No keyword matches.")

    print("\nSemantic search results")
    semantic_results = semantic_search(query, top_k=top_k)

    for doc_id, doc, score in semantic_results:
        print(f"{doc_id}. {doc}  (score = {score:.2f})")

In [13]:
compare_search("puppy")
compare_search("doctor")
compare_search("bike")
compare_search("storm")
compare_search("songs")


QUERY: 'puppy'

Keyword search results
1. My puppy loves chasing tennis balls.  (score = 2.17)

Semantic search results
1. My puppy loves chasing tennis balls.  (score = 0.46)
2. My dog enjoys long walks in the park.  (score = 0.27)
9. Taylor Swift released a new album.  (score = 0.20)
14. The doctor prescribed antibiotics.  (score = 0.20)
15. Penguins cannot fly.  (score = 0.19)

QUERY: 'doctor'

Keyword search results
14. The doctor prescribed antibiotics.  (score = 2.58)

Semantic search results
14. The doctor prescribed antibiotics.  (score = 0.58)
13. The physician recommended more exercise.  (score = 0.38)
11. I rode my bicycle to school.  (score = 0.16)
10. I listened to some great music.  (score = 0.15)
9. Taylor Swift released a new album.  (score = 0.13)

QUERY: 'bike'

Keyword search results
12. My bike has a flat tire.  (score = 2.17)

Semantic search results
11. I rode my bicycle to school.  (score = 0.58)
12. My bike has a flat tire.  (score = 0.45)
14. The doctor prescr